In [1]:
import pandas as pd

SEED = 67

queries = pd.read_json("raw/queries.jsonl", lines=True)
queries.drop('metadata', axis=1, inplace=True)
queries.columns = ['Request_id', 'text']

apis = pd.read_json("raw/corpus.jsonl", lines=True)
apis.drop(['title', 'metadata'], axis=1, inplace=True)
apis.columns = ['API_id', 'description']

interactions = pd.read_csv("raw/query_tool.txt", sep='\t', names=['Request_id', 'API_id'])

In [2]:
if len(queries) == queries['Request_id'].unique().size:
    print('Все Request_id уникальные')
else:
    print('Есть дублирующиеся Request_id')

Все Request_id уникальные


In [3]:
from sklearn.model_selection import train_test_split

train_ids, remaining_ids = train_test_split(
    queries["Request_id"], test_size=0.2, random_state=SEED
)

val_ids, test_ids = train_test_split(
    remaining_ids, test_size=0.5, random_state=SEED
)

id_to_split = {
    **dict.fromkeys(train_ids, "train"),
    **dict.fromkeys(val_ids, "val"),
    **dict.fromkeys(test_ids, "test"),
}

queries["split"] = queries["Request_id"].map(id_to_split)

print(queries["split"].value_counts())

split
train    15016
val       1877
test      1877
Name: count, dtype: int64


In [4]:
queries.head()

,Request_id,text,split
0,18674,I'm planning a meal using the ingredient beef ...,train
1,13239,I'm looking for dinner ideas using the ingredi...,val
2,10855,I'm discovering recipes using the ingredient t...,train
3,15051,I'm exploring vegan options using the ingredie...,train
4,14920,I'm preparing quick snacks using the ingredien...,train


In [5]:
queries.info()

<class 'pandas.DataFrame'>
RangeIndex: 18770 entries, 0 to 18769
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Request_id  18770 non-null  int64
 1   text        18770 non-null  str  
 2   split       18770 non-null  str  
dtypes: int64(1), str(2)
memory usage: 3.0 MB


In [6]:
apis.head()

,API_id,description
0,0,"category_name:Food, tool_name:Worldwide Recipe..."
1,1,"category_name:Food, tool_name:Nutrition by API..."
2,2,"category_name:Food, tool_name:Store Groceries,..."
3,3,"category_name:Finance, tool_name:Real-Time Fin..."
4,4,"category_name:Finance, tool_name:Metals Prices..."


In [7]:
apis.info()

<class 'pandas.DataFrame'>
RangeIndex: 464 entries, 0 to 463
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   API_id       464 non-null    int64
 1   description  464 non-null    str  
dtypes: int64(1), str(1)
memory usage: 212.3 KB


In [8]:
interactions = interactions.drop_duplicates(
    subset=["Request_id", "API_id"]
).reset_index(drop=True)
interactions['relevance'] = 1.0

In [9]:
interactions.head()

,Request_id,API_id,relevance
0,18674,0,1.0
1,18674,1,1.0
2,18674,2,1.0
3,13239,0,1.0
4,13239,1,1.0


In [10]:
interactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 49852 entries, 0 to 49851
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Request_id  49852 non-null  int64  
 1   API_id      49852 non-null  int64  
 2   relevance   49852 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 1.1 MB


In [11]:
output_dir = "processed"

queries.to_parquet(f"{output_dir}/requests.parquet", index=False)
apis.to_parquet(f"{output_dir}/apis.parquet", index=False)
interactions.to_parquet(f"{output_dir}/interactions.parquet", index=False)